In [1]:
from dnallm import load_config
from dnallm.models import load_model_and_tokenizer
from dnallm.inference import DNAInference

In [2]:
# Provenance stamp (D-21): the exact versions this notebook ran with,
# printed as key=value lines from the real execution environment
# (first real execution: 08-06, isolated dnallm-evo kernel). The evo-1
# leg runs the 8k variant and the evo2 leg the noFP8 config path
# (05-FEASIBILITY verdicts; see the capability-override cell below).
import torch
import transformers
import flash_attn

print(f"transformers_version={transformers.__version__}")
print(f"torch_version={torch.__version__}")
print(f"flash_attn_version={flash_attn.__version__}")


transformers_version=5.18.0

torch_version=2.11.0+cu130

flash_attn_version=2.8.3.post1

In [3]:
# Load configurations
configs = load_config("./inference_evo_config.yaml")

## EVO2

In [4]:
## Make sure you have installed the dependencies for Evo2 model
## (FEASIBILITY lock: evo2==0.3.0 pulls vtx; transformer-engine must STAY
##  ABSENT -- the empty TE meta package breaks `from evo2 import Evo2` and
##  the noFP8 config below makes TE unnecessary; flash-attn compile below)
# !uv pip install "flash_attn==2.8.3.post1" --no-build-isolation --no-cache-dir
# !uv pip install evo2==0.3.0


In [5]:
# Force the noFP8 config path (05-FEASIBILITY verdict): this device
# reports FP8 capability (compute capability >= 9.0), which would
# auto-select the FP8 config requiring Transformer Engine -- TE is
# deliberately absent because the empty TE meta package raises at
# import and breaks `from evo2 import Evo2`. With flash-attn present
# the handler resolves evo2-1b-8k-noFP8.yml (use_fp8_input_projections=False).
import dnallm.models.special.evo as _evo_mod


def _no_fp8() -> bool:
    """Documented deviation: the noFP8 config is mandatory here."""
    return False


_evo_mod.is_fp8_capable = _no_fp8


In [6]:
model_name = "arcinstitute/evo2_1b_base"
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs['task'], source="huggingface")

07:44:26 - dnallm.utils.support - 

WARNING

 - Generation task does not require num_labels, but got 1. Setting to 0.


07:44:26 - dnallm.utils.support - 

INFO

...

100%|██████████| 25/25 [00:00<00:00, 90.93it/s]

                    INFO     StripedHyena - INFO - Initialized model                                   ]8;id=5434232;file:///home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=5434233;file:///home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/model.py#723\723]8;;\

                    INFO     vortex.model.utils - INFO - Loading                                        ]8;id=5434240;file:///home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/utils.py\utils.py]8;;\:]8;id=5434241;file:///home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/utils.py#92\92]8;;\
                             /home/forrest/models-giants/hub/models--arcinstitute--evo2_1b_base/snapsho            
                             ts/2279e1df422c991037470302360edd40d0d2ea1e/evo2_1b_base.pt                           

Extra keys in state_dict: {'blocks.20.projections._extra_state', 'blocks.12.projections._extra_state', 'blocks.6.projections._extra_state', 'blocks.13.projections._extra_state', 'blocks.14.projections._extra_state', 'blocks.11.projections._extra_state', 'blocks.17.mixer.dense._extra_state', 'blocks.7.projections._extra_state', 'blocks.23.projections._extra_state', 'blocks.4.projections._extra_state', 'blocks.8.projections._extra_state', 'blocks.2.projections._extra_state', 'blocks.15.projections._extra_state', 'blocks.5.projections._extra_state', 'unembed.weight', 'blocks.24.mixer.dense._extra_state', 'blocks.0.projections._extra_state', 'blocks.10.mixer.dense._extra_state', 'blocks.21.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.22.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.1.projections._extra_state', 'blocks.16.projections._extra_state', 'blocks.9.projections._extra_state', 'blocks.3.mixer.dense._extra_state'}

[10/07/26 07:44:43] INFO     StripedHyena - INFO - Adjusting Wqkv for column split (permuting rows)   ]8;id=5434247;file:///home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=5434248;file:///home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/model.py#1008\1008]8;;\

In [7]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

07:44:43 - dnallm.utils.support - 

INFO

 - Using device: cuda


                    INFO dnallm.utils.support - [36mINFO[0m - Using device: cuda                ]8;id=5434253;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py\logger.py]8;;\:]8;id=5434254;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py#75\75]8;;\

In [8]:
output = inference_engine.generate(["@", "ATG"])

Initializing inference params with max_seqlen=401

/home/forrest/Github/DNALLM/.scratch/evo-venvs/evo/lib/python3.13/site-packages/vortex/model/engine.py:616: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /pytorch/aten/src/ATen/native/Copy.cpp:308.)
  inference_params.state_dict[layer_idx] = state[..., L - 1].to(dtype=state_dtype)


Initializing inference params with max_seqlen=403

Prompt: "@",	Output: "TCCATTGTACGAATTGATTTTCAAGTGCCGCAAAGTAATCATTCATTTTTTCAGAATTTGAAGAGCGAACATATATTAGGACTATTTTTTTTCCAAAAAATTTGTAAAATAAAGCGAACATATCACAGAGGGCGTCTCAAACCGAATTTTACCGGTCTATTCTGCAAAGCAAAAATTTTCCCCCCAATCTCGCTGCAAAACGCAAAATCTTACCTTCGTCAGTTCTACTTCACTACTTCATTGATAATCCTTGCCTTGATCTAATGGTAAACTCTTGTATCCGTACAACAACATGTGAGGACAGTATCGATGATCAAATACCATCCAATTCAAGACTACGACCTATAGAGTAGAATTTCATGCTTACGAGAAGCTCAATATATAATGGAGGATCGTACTC",	Score: -1.3546857833862305

Prompt: "ATG",	Output: "GTACCTTACGCCAGGTACCACTCCTCAAATTTCATTCCCCGAGACAAGCAGAAGGTAATAATTTGCTATCATAAAATAGTGGGACCACTAGTTGCATCTCCCTGTAACAGGGCATTCTCCTTTCTCCAGGCCAGATGACCTTCGAACTCCGCGTATAAAGGCGGACCCGCATCAAAAAACGCAGCGCTTATTCGGATAATATTATTTGGATTTAATCAAATACTGACTTGACTTTCTTACAGAAACCATCCTCCTGACTTACCAAAGCTATGAGACCTCCTTGACAACTGGTCCATTGACTACCCTACCGCGTAGCTGGTAGACTAGTCGGATCGCTAATCTCCTTAACATCTATTGACTTCACAGCTGAAAATGAGTGCCACCTGCTTCCGAGCAACTC",	Score: -1.3602713346481323

In [9]:
for seq in output:
    print(f"Input Sequence: {seq['Prompt']}")
    print(f"Generated Sequence: {seq['Output']}")
    print(f"Score: {seq['Score']}")
    print()

Input Sequence: @

Generated Sequence: TCCATTGTACGAATTGATTTTCAAGTGCCGCAAAGTAATCATTCATTTTTTCAGAATTTGAAGAGCGAACATATATTAGGACTATTTTTTTTCCAAAAAATTTGTAAAATAAAGCGAACATATCACAGAGGGCGTCTCAAACCGAATTTTACCGGTCTATTCTGCAAAGCAAAAATTTTCCCCCCAATCTCGCTGCAAAACGCAAAATCTTACCTTCGTCAGTTCTACTTCACTACTTCATTGATAATCCTTGCCTTGATCTAATGGTAAACTCTTGTATCCGTACAACAACATGTGAGGACAGTATCGATGATCAAATACCATCCAATTCAAGACTACGACCTATAGAGTAGAATTTCATGCTTACGAGAAGCTCAATATATAATGGAGGATCGTACTC

Score: -1.3546857833862305

Input Sequence: ATG

Generated Sequence: GTACCTTACGCCAGGTACCACTCCTCAAATTTCATTCCCCGAGACAAGCAGAAGGTAATAATTTGCTATCATAAAATAGTGGGACCACTAGTTGCATCTCCCTGTAACAGGGCATTCTCCTTTCTCCAGGCCAGATGACCTTCGAACTCCGCGTATAAAGGCGGACCCGCATCAAAAAACGCAGCGCTTATTCGGATAATATTATTTGGATTTAATCAAATACTGACTTGACTTTCTTACAGAAACCATCCTCCTGACTTACCAAAGCTATGAGACCTCCTTGACAACTGGTCCATTGACTACCCTACCGCGTAGCTGGTAGACTAGTCGGATCGCTAATCTCCTTAACATCTATTGACTTCACAGCTGAAAATGAGTGCCACCTGCTTCCGAGCAACTC

Score: -1.3602713346481323

In [10]:
scores = inference_engine.scoring(["ATCCGCATG", "ATGCGCATG"])
for res in scores:
    print(f"Input Sequence: {res['Input']}")
    print(f"Score: {res['Score']}")
    print()

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 64.58it/s]

Input Sequence: ATCCGCATG

Score: -1.4140625

Input Sequence: ATGCGCATG

Score: -1.4091796875

## EVO1

In [11]:
## Make sure you have installed the dependencies for Evo-1 model
## (FEASIBILITY-locked pins, 08-06: evo-model 0.5 + stripedhyena 0.2.2 --no-deps
##  + flash-attn 2.8.3.post1; evo-1-8k remote code also needs the numpy-2
##  np.fromstring shim dnallm ships in transformers_compat)
# !uv pip install evo-model==0.5
# !uv pip install --no-deps stripedhyena==0.2.2
# !uv pip install "flash_attn==2.8.3.post1" --no-build-isolation --no-cache-dir


In [12]:
# Force the noFA config path for the evo-1 load (CR-01 gate-4 evidence):
# this box's evo venv flash_attn 2.8.3.post1 wheel imports under
# torch 2.11.0+cu130, but its C++ op has no registered schema -- the first
# flash_attn_kvpacked_func call deterministically raises the OperatorEntry
# INTERNAL ASSERT, and the import-only capability probe cannot see that.
# Patched off, the handler resolves evo-1-8k-base-noFA.yml (same deviation
# pattern as the noFP8 force above; the evo2 section already ran green).
import dnallm.models.special.evo as _evo_mod


def _no_flash_attention() -> bool:
    """Documented deviation: the noFA config is mandatory for evo-1 here."""
    return False


_evo_mod.is_flash_attention_capable = _no_flash_attention

In [13]:
model_name = "togethercomputer/evo-1-8k-base"
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs['task'], source="huggingface")

07:44:57 - dnallm.utils.support - 

WARNING

 - Generation task does not require num_labels, but got 1. Setting to 0.


[10/07/26 07:44:57] WARNING dnallm.utils.support - [33mWARNING[0m - Generation task does not    ]8;id=5434260;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py\logger.py]8;;\:]8;id=5434261;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py#79\79]8;;\
                                   require num_labels, but got 1. Setting to 0.                                    

07:44:57 - dnallm.utils.support - 

INFO

 - Model files are stored in /home/forrest/models-giants/hub/models--togethercomputer--evo-1-8k-base/snapshots/a9be7b66485080893399ade87c7d34f81ad3e249


                    INFO dnallm.utils.support - [36mINFO[0m - Model files are stored in         ]8;id=5434266;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py\logger.py]8;;\:]8;id=5434267;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py#75\75]8;;\
                                /home/forrest/models-giants/hub/models--togethercomputer--evo-1-8k-bas             
                                e/snapshots/a9be7b66485080893399ade87c7d34f81ad3e249                               

Loading weights:   0%|          | 0/438 [00:00<?, ?it/s]

In [14]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

07:45:11 - dnallm.utils.support - 

INFO

 - Using device: cuda


[10/07/26 07:45:11] INFO dnallm.utils.support - [36mINFO[0m - Using device: cuda                ]8;id=5434272;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py\logger.py]8;;\:]8;id=5434273;file:///home/forrest/Github/DNALLM/dnallm/utils/logger.py#75\75]8;;\

In [15]:
output = inference_engine.generate(["@", "ACGT"])

Note: Prompts are of different lengths.


Note: Will not do batched generation.


Prompt: "@",	Output: "TACCCCGCACCGGGGCAGGCAGAGGATAAAAATATCTGCAAGCAAAATGCAATAAAAAGGCAAGAGAAACACATTGTTAAATTTTTTTACTAACGCCACTAGAAAAACATTTTGTATTCAATTAAATTAGAAATGGCCTAGGCAGGGCCACATTTTTTTGTTTTTGGAAAAGGGGGAAAAGAAAGGAAACAGCGGAGGGCCCAATAAAAGCCCGTTGGGTGTTTTTTATGCACCCGAGAATCATTTTATAACAATGAAAAAAGCGGGTCTTCGAAGAGATCGTGGGAAATTTGCGTCGACTACTACAATTTTGCTCCTGACCAGCAAAAAAAAAAACATCTGGGACGAATCTAGAGTTCTCAAAATAATGTAGTATCAATTGCAACAAACACCGTCCGAA",	Score: -1.3749008178710938

Prompt: "ACGT",	Output: "CCCCCGTCGGAAGATCGGCACGATTTCACCAAGATCTGCTGCGTTCAGGAATAGTCTCTCAAAACAAAAAATCCCTTAAACAAAAGCATCCCGACCGGGACGAAGAATTATTCCTCATAACTAAATAAAAGAAATAGCGACACACGAATCTCACATTAGGGATTAGGACATATCAGTTCTTTTTATATGAAAAAATCTTATCTTTTTCATAAATATACAATGACCGTATATCATTATTAAATGATAATAATTAACACTCTTCATAGTTATCATTATCTAGCGGATTTATTTCATTTATTTCTTCTTGAGTTACTAAGACTAAATCTTTGGGTAAACCACTACGCTCAACGTTCTTAATATAATTTTCCCGTTCCTTTTTTTTATACGTTTTTTTTTCATT",	Score: -1.589525818824768

In [16]:
for seq in output:
    print(f"Input Sequence: {seq['Prompt']}")
    print(f"Generated Sequence: {seq['Output']}")
    print(f"Score: {seq['Score']}")
    print()

Input Sequence: @

Generated Sequence: TACCCCGCACCGGGGCAGGCAGAGGATAAAAATATCTGCAAGCAAAATGCAATAAAAAGGCAAGAGAAACACATTGTTAAATTTTTTTACTAACGCCACTAGAAAAACATTTTGTATTCAATTAAATTAGAAATGGCCTAGGCAGGGCCACATTTTTTTGTTTTTGGAAAAGGGGGAAAAGAAAGGAAACAGCGGAGGGCCCAATAAAAGCCCGTTGGGTGTTTTTTATGCACCCGAGAATCATTTTATAACAATGAAAAAAGCGGGTCTTCGAAGAGATCGTGGGAAATTTGCGTCGACTACTACAATTTTGCTCCTGACCAGCAAAAAAAAAAACATCTGGGACGAATCTAGAGTTCTCAAAATAATGTAGTATCAATTGCAACAAACACCGTCCGAA

Score: -1.3749008178710938

Input Sequence: ACGT

Generated Sequence: CCCCCGTCGGAAGATCGGCACGATTTCACCAAGATCTGCTGCGTTCAGGAATAGTCTCTCAAAACAAAAAATCCCTTAAACAAAAGCATCCCGACCGGGACGAAGAATTATTCCTCATAACTAAATAAAAGAAATAGCGACACACGAATCTCACATTAGGGATTAGGACATATCAGTTCTTTTTATATGAAAAAATCTTATCTTTTTCATAAATATACAATGACCGTATATCATTATTAAATGATAATAATTAACACTCTTCATAGTTATCATTATCTAGCGGATTTATTTCATTTATTTCTTCTTGAGTTACTAAGACTAAATCTTTGGGTAAACCACTACGCTCAACGTTCTTAATATAATTTTCCCGTTCCTTTTTTTTATACGTTTTTTTTTCATT

Score: -1.589525818824768

In [17]:
scores = inference_engine.scoring(["ATCCGCATG", "ATGCGCATG"])
for res in scores:
    print(f"Input Sequence: {res['Input']}")
    print(f"Score: {res['Score']}")
    print()

Input Sequence: ATCCGCATG

Score: -1.4131944179534912

Input Sequence: ATGCGCATG

Score: -1.3697916269302368